# Notebook 08 - Fast Ensemble Kaggle (Phase 2)

But: generer plusieurs soumissions rapides en fusionnant des classements deja calcules.

Ce notebook est utile pour tester vite plusieurs variantes sur Kaggle sans recalcul lourd.


In [1]:
import json
import os
from collections import defaultdict

import pandas as pd

CHAT_OUTPUTS_DIR = '../outputs'
VAL_OUTPUTS_DIR = '../../val/outputs'
K_SUBMIT = 100

print('CHAT_OUTPUTS_DIR =', CHAT_OUTPUTS_DIR)
print('VAL_OUTPUTS_DIR  =', VAL_OUTPUTS_DIR)


CHAT_OUTPUTS_DIR = ../outputs
VAL_OUTPUTS_DIR  = ../../val/outputs


In [2]:
RUN_FILES = {
    # runs forts provenant des experimentations avancees
    'v8_hybrid_cv': os.path.join(VAL_OUTPUTS_DIR, '92iemj_v8_hybrid_cv.csv'),
    'phase2_final': os.path.join(VAL_OUTPUTS_DIR, 'submission_phase2_final.csv'),
    'v7_tuned': os.path.join(VAL_OUTPUTS_DIR, '92iemj_v7_tuned.csv'),
    'v5_cross': os.path.join(VAL_OUTPUTS_DIR, '92iemj_v5_crossencoder.csv'),
    'baseline_92': os.path.join(VAL_OUTPUTS_DIR, '92iemj.csv'),

    # run genere dans chatodit
    'our_v1': os.path.join(CHAT_OUTPUTS_DIR, 'phase2_submission_v1.csv'),
}

CATEGORY_SOURCE = 'phase2_final'


def load_submission(path):
    df = pd.read_csv(path)
    expected = {'query_id', 'relevant_doc_ids', 'category'}
    missing = expected - set(df.columns)
    if missing:
        raise ValueError(f'{path} missing columns: {missing}')
    df = df.copy()
    df['relevant_doc_ids'] = df['relevant_doc_ids'].apply(json.loads)
    return df


loaded = {}
for run_name, path in RUN_FILES.items():
    if not os.path.exists(path):
        print(f'[WARN] missing: {path}')
        continue
    loaded[run_name] = load_submission(path)
    print(f"[OK] {run_name:12s} -> {os.path.basename(path)} ({len(loaded[run_name])} queries)")

if not loaded:
    raise RuntimeError('No candidate submissions found.')

# Integrity check: same query order
base_name = next(iter(loaded))
base_ids = loaded[base_name]['query_id'].tolist()
for run_name, df_run in loaded.items():
    if df_run['query_id'].tolist() != base_ids:
        raise ValueError(f'Query order mismatch for run: {run_name}')

cat_map = dict(zip(loaded[CATEGORY_SOURCE]['query_id'], loaded[CATEGORY_SOURCE]['category']))
print('Category source =', CATEGORY_SOURCE)


[OK] v8_hybrid_cv -> 92iemj_v8_hybrid_cv.csv (141 queries)
[OK] phase2_final -> submission_phase2_final.csv (141 queries)
[OK] v7_tuned     -> 92iemj_v7_tuned.csv (141 queries)
[OK] v5_cross     -> 92iemj_v5_crossencoder.csv (141 queries)
[OK] baseline_92  -> 92iemj.csv (141 queries)
[OK] our_v1       -> phase2_submission_v1.csv (141 queries)
Category source = phase2_final


In [3]:
def weighted_rrf(rankings, weights, k_rrf=60, topk=100):
    scores = defaultdict(float)
    for docs, w in zip(rankings, weights):
        for rank, doc_id in enumerate(docs, start=1):
            scores[doc_id] += float(w) * (1.0 / (k_rrf + rank))
    fused = sorted(scores.keys(), key=lambda d: scores[d], reverse=True)
    return fused[:topk]


CONFIGS = [
    {
        'name': 'phase2_ens_v8_p2f_equal.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final'],
        'weights': [1.0, 1.0],
        'k_rrf': 60,
    },
    {
        'name': 'phase2_ens_v8_p2f_2to1.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final'],
        'weights': [2.0, 1.0],
        'k_rrf': 20,
    },
    {
        'name': 'phase2_ens_v8_p2f_v7.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v7_tuned'],
        'weights': [2.0, 1.0, 0.5],
        'k_rrf': 20,
    },
    {
        'name': 'phase2_ens_all4.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v7_tuned', 'baseline_92'],
        'weights': [2.0, 1.0, 0.5, 0.5],
        'k_rrf': 20,
    },
    {
        'name': 'phase2_ens_plus_cross.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v5_cross', 'our_v1'],
        'weights': [1.5, 1.0, 1.0, 0.5],
        'k_rrf': 20,
    },
    {
        'name': 'phase2_ens_plus_cross_strong.csv',
        'runs': ['v8_hybrid_cv', 'phase2_final', 'v5_cross', 'our_v1', 'v7_tuned'],
        'weights': [2.0, 1.2, 1.0, 0.5, 0.5],
        'k_rrf': 10,
    },
]

for cfg in CONFIGS:
    missing_runs = [r for r in cfg['runs'] if r not in loaded]
    if missing_runs:
        print(f"[SKIP] {cfg['name']} missing runs: {missing_runs}")
        continue

    rows = []
    qids = loaded[cfg['runs'][0]]['query_id'].tolist()

    for i, qid in enumerate(qids):
        rankings = [loaded[r].iloc[i]['relevant_doc_ids'] for r in cfg['runs']]
        fused = weighted_rrf(rankings, cfg['weights'], k_rrf=cfg['k_rrf'], topk=K_SUBMIT)

        rows.append({
            'query_id': qid,
            'relevant_doc_ids': json.dumps(fused),
            'category': cat_map.get(qid, 'tex'),
        })

    df_out = pd.DataFrame(rows)
    n_docs = df_out['relevant_doc_ids'].apply(lambda x: len(json.loads(x)))
    assert (n_docs == K_SUBMIT).all(), f'Invalid top-k in {cfg["name"]}'

    out_path = os.path.join(CHAT_OUTPUTS_DIR, cfg['name'])
    df_out.to_csv(out_path, index=False)
    print(f"[OK] {cfg['name']} | rows={len(df_out)}")


[OK] phase2_ens_v8_p2f_equal.csv | rows=141
[OK] phase2_ens_v8_p2f_2to1.csv | rows=141
[OK] phase2_ens_v8_p2f_v7.csv | rows=141
[OK] phase2_ens_all4.csv | rows=141
[OK] phase2_ens_plus_cross.csv | rows=141
[OK] phase2_ens_plus_cross_strong.csv | rows=141


## Prochaines soumissions a tester (ordre recommande)

1. `phase2_ens_v8_p2f_2to1.csv`
2. `phase2_ens_plus_cross_strong.csv`
3. `phase2_ens_v8_p2f_equal.csv`
4. `phase2_ens_plus_cross.csv`
5. `phase2_ens_v8_p2f_v7.csv`
6. `phase2_ens_all4.csv`
